In [ ]:
from __future__ import annotations
import os
# Limit engine threads so outer parallelism (6) x inner parallelism (4) uses 24 worker threads, about 75% of a 32-core machine.
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'
os.environ['LGBM_DEFAULT_NUM_THREADS'] = '4'
import csv
import sys

In [ ]:
from __future__ import annotations

import csv
import sys
import time
import traceback
import uuid
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


OUTER_FOLDS = (1, 2, 3, 4, 5)
ML_BASELINE_MODEL_SPECS = [
    ("ml_et", "et"),
    ("ml_xgboost", "xgboost"),
    ("ml_catboost", "catboost"),
    ("ml_rf", "rf"),
    ("ml_lightgbm", "lightgbm"),
]
ML_TOP_K_TUNED = 2
TARGET_COL = "Target_Log"
RUNTIME_PROGRESS_LOG_NAME = "1_ml_run_progress.log"
RUNTIME_STATUS_FILE_NAME = "1_ml_run_status.csv"

build_prediction_frame = None
build_pycaret_setup_kwargs = None
compute_regression_metrics = None
ensure_family_result_dir = None
load_full_feature_processed = None
save_model_artifact = None
summarize_fold_metrics = None
setup = None
create_model = None
tune_model = None
finalize_model = None
predict_model = None
pull = None
ACTIVE_ML_RUN_OBSERVER = None


In [ ]:
def resolve_current_code_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    code_dir = Path.cwd().resolve()
    if code_dir.name != "1_Code":
        raise RuntimeError(
            "cannot resolve current code directory without __file__; "
            "current working directory must be the Step5 1_Code directory"
        )
    return code_dir


In [ ]:
def ensure_pycaret_runtime_compat() -> None:
    import numpy as np
    import scipy
    import sklearn.utils as sklearn_utils

    if not hasattr(sklearn_utils, "_print_elapsed_time"):
        @contextmanager
        def _print_elapsed_time(*args, **kwargs):
            del args, kwargs
            yield

        sklearn_utils._print_elapsed_time = _print_elapsed_time

    if not hasattr(scipy, "interp"):
        scipy.interp = np.interp


In [ ]:
def initialize_runtime() -> Path:
    global TARGET_COL
    global build_prediction_frame
    global build_pycaret_setup_kwargs
    global compute_regression_metrics
    global ensure_family_result_dir
    global load_full_feature_processed
    global save_model_artifact
    global summarize_fold_metrics
    global setup
    global create_model
    global tune_model
    global finalize_model
    global predict_model
    global pull

    current_code_dir = resolve_current_code_dir()
    if str(current_code_dir) not in sys.path:
        sys.path.insert(0, str(current_code_dir))

    ensure_pycaret_runtime_compat()

    try:
        from pycaret.regression import (
            create_model as pycaret_create_model,
            finalize_model as pycaret_finalize_model,
            predict_model as pycaret_predict_model,
            pull as pycaret_pull,
            setup as pycaret_setup,
            tune_model as pycaret_tune_model,
        )
    except Exception as exc:  # pragma: no cover - runtime fallback for environments without pycaret
        def _missing(*args, **kwargs):
            raise RuntimeError("pycaret.regression is unavailable") from exc

        pycaret_setup = _missing
        pycaret_create_model = _missing
        pycaret_tune_model = _missing
        pycaret_finalize_model = _missing
        pycaret_predict_model = _missing
        pycaret_pull = _missing

    from _step5_shared import (
        TARGET_COL as shared_target_col,
        build_prediction_frame as shared_build_prediction_frame,
        build_pycaret_setup_kwargs as shared_build_pycaret_setup_kwargs,
        compute_regression_metrics as shared_compute_regression_metrics,
        ensure_family_result_dir as shared_ensure_family_result_dir,
        load_full_feature_processed as shared_load_full_feature_processed,
        save_model_artifact as shared_save_model_artifact,
        summarize_fold_metrics as shared_summarize_fold_metrics,
    )

    TARGET_COL = shared_target_col
    if build_prediction_frame is None:
        build_prediction_frame = shared_build_prediction_frame
    if build_pycaret_setup_kwargs is None:
        build_pycaret_setup_kwargs = shared_build_pycaret_setup_kwargs
    if compute_regression_metrics is None:
        compute_regression_metrics = shared_compute_regression_metrics
    if ensure_family_result_dir is None:
        ensure_family_result_dir = shared_ensure_family_result_dir
    if load_full_feature_processed is None:
        load_full_feature_processed = shared_load_full_feature_processed
    if save_model_artifact is None:
        save_model_artifact = shared_save_model_artifact
    if summarize_fold_metrics is None:
        summarize_fold_metrics = shared_summarize_fold_metrics
    if setup is None:
        setup = pycaret_setup
    if create_model is None:
        create_model = pycaret_create_model
    if tune_model is None:
        tune_model = pycaret_tune_model
    if finalize_model is None:
        finalize_model = pycaret_finalize_model
    if predict_model is None:
        predict_model = pycaret_predict_model
    if pull is None:
        pull = pycaret_pull
    return current_code_dir


In [ ]:
def expected_output_files() -> list[str]:
    return [
        "1_ml_model_artifact_manifest.csv",
        "1_ml_predictions_outer_test.csv",
        "1_ml_metrics_by_model_by_outer_fold.csv",
        "1_ml_metrics_by_model_mean_std.csv",
        "1_ml_model_ranking.csv",
        "1_ml_best_models.csv",
    ]


In [ ]:
def _utc_now_text() -> str:
    return datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


def _format_seconds(seconds: float | None) -> str:
    if seconds is None:
        return ""
    return f"{float(seconds):.2f}s"


class MLRunObserver:
    def __init__(self, result_dir: Path, total_units: int):
        self.result_dir = Path(result_dir)
        self.total_units = int(total_units)
        self.run_id = uuid.uuid4().hex[:12]
        self.started_at = _utc_now_text()
        self.completed_units = 0
        self.log_path = self.result_dir / RUNTIME_PROGRESS_LOG_NAME
        self.status_path = self.result_dir / RUNTIME_STATUS_FILE_NAME
        self.progress_bar = tqdm(
            total=self.total_units,
            desc="ml_family_progress",
            unit="unit",
            file=sys.stdout,
        )

    def _append_log_line(self, text: str) -> None:
        self.result_dir.mkdir(parents=True, exist_ok=True)
        with self.log_path.open("a", encoding="utf-8") as fh:
            fh.write(text + "\n")

    def _write_status_row(self, row: dict[str, object]) -> None:
        fieldnames = [
            "run_id",
            "started_at",
            "updated_at",
            "outer_fold",
            "model_id",
            "run_role",
            "stage",
            "trial_index",
            "trial_total",
            "completed_units",
            "total_units",
            "pct_complete",
            "status",
        ]
        self.result_dir.mkdir(parents=True, exist_ok=True)
        with self.status_path.open("w", newline="", encoding="utf-8") as fh:
            writer = csv.DictWriter(fh, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerow(row)

    def start_run(self, step4_root: Path, step5_root: Path) -> None:
        self._append_log_line(
            f"[{_utc_now_text()}] RUN START | run_id={self.run_id} | step4_root={step4_root} | step5_root={step5_root}"
        )
        self.update_status(
            outer_fold=None,
            model_id=None,
            run_role=None,
            stage="run_start",
            trial_index=None,
            trial_total=None,
            completed_units=0,
            status="running",
        )
        print(f"progress: 0/{self.total_units} run started")

    def log_stage(
        self,
        event: str,
        outer_fold: int | None,
        model_id: str | None,
        run_role: str | None,
        stage: str,
        message: str,
        trial_index: int | None = None,
        trial_total: int | None = None,
        elapsed_sec: float | None = None,
    ) -> None:
        line = (
            f"[{_utc_now_text()}] {event} | outer_fold={outer_fold} | model_id={model_id} | "
            f"run_role={run_role} | stage={stage} | trial_index={trial_index} | trial_total={trial_total} | "
            f"elapsed={_format_seconds(elapsed_sec)} | {message}"
        )
        self._append_log_line(line)

    def update_status(
        self,
        outer_fold: int | None,
        model_id: str | None,
        run_role: str | None,
        stage: str,
        trial_index: int | None,
        trial_total: int | None,
        completed_units: int,
        status: str,
    ) -> None:
        pct_complete = round((float(completed_units) / max(float(self.total_units), 1.0)) * 100.0, 2)
        self.completed_units = int(completed_units)
        self._write_status_row(
            {
                "run_id": self.run_id,
                "started_at": self.started_at,
                "updated_at": _utc_now_text(),
                "outer_fold": outer_fold,
                "model_id": model_id,
                "run_role": run_role,
                "stage": stage,
                "trial_index": trial_index,
                "trial_total": trial_total,
                "completed_units": int(completed_units),
                "total_units": int(self.total_units),
                "pct_complete": pct_complete,
                "status": status,
            }
        )

    def advance_unit(self, message: str) -> None:
        self.completed_units += 1
        self.progress_bar.update(1)
        self.progress_bar.set_postfix_str(message)
        print(f"progress: {self.completed_units}/{self.total_units} {message}")

    def finish_run(self, output_files: list[str], total_elapsed_sec: float) -> None:
        self._append_log_line(
            f"[{_utc_now_text()}] RUN COMPLETED | run_id={self.run_id} | elapsed={_format_seconds(total_elapsed_sec)} | outputs={output_files}"
        )
        self.update_status(
            outer_fold=None,
            model_id=None,
            run_role=None,
            stage="run_completed",
            trial_index=None,
            trial_total=None,
            completed_units=self.total_units,
            status="completed",
        )
        self.progress_bar.close()

    def fail_run(
        self,
        outer_fold: int | None,
        model_id: str | None,
        run_role: str | None,
        stage: str,
        exc: Exception,
    ) -> None:
        self._append_log_line(
            f"[{_utc_now_text()}] RUN FAILED | outer_fold={outer_fold} | model_id={model_id} | run_role={run_role} | "
            f"stage={stage} | exc_type={type(exc).__name__} | exc={exc}"
        )
        self._append_log_line(traceback.format_exc())
        self.update_status(
            outer_fold=outer_fold,
            model_id=model_id,
            run_role=run_role,
            stage=stage,
            trial_index=None,
            trial_total=None,
            completed_units=self.completed_units,
            status="failed",
        )
        self.progress_bar.close()


def create_ml_run_observer(result_dir: Path, total_units: int) -> MLRunObserver:
    return MLRunObserver(result_dir=result_dir, total_units=total_units)


In [ ]:
def log_ml_mainline_outer_fold_start(outer_fold: int) -> None:
    observer = ACTIVE_ML_RUN_OBSERVER
    if observer is None:
        return
    observer.log_stage(
        event="outer_fold_start",
        outer_fold=int(outer_fold),
        model_id=None,
        run_role=None,
        stage="outer_fold_start",
        message="starting outer fold",
    )
    observer.update_status(
        outer_fold=int(outer_fold),
        model_id=None,
        run_role=None,
        stage="outer_fold_start",
        trial_index=None,
        trial_total=None,
        completed_units=observer.completed_units,
        status="running",
    )


def log_ml_mainline_tuned_candidates_selected(outer_fold: int, baseline_ranking: pd.DataFrame) -> None:
    observer = ACTIVE_ML_RUN_OBSERVER
    if observer is None:
        return
    tuned_candidate_df = pd.DataFrame(baseline_ranking).head(int(ML_TOP_K_TUNED)).reset_index(drop=True)
    observer.log_stage(
        event="tuned_candidates_selected",
        outer_fold=int(outer_fold),
        model_id=",".join(tuned_candidate_df["model_id"].astype(str).tolist()),
        run_role="tuned",
        stage="tuned_candidates_selected",
        message="selected tuned candidates from baseline ranking",
    )


def finalize_ml_mainline_run(
    result_dir: Path,
    prediction_frames: list[pd.DataFrame],
    metric_rows: list[dict[str, object]],
    artifact_rows: list[dict[str, object]],
    run_start: float,
) -> None:
    write_ml_family_outputs(result_dir, prediction_frames, metric_rows, artifact_rows)
    observer = ACTIVE_ML_RUN_OBSERVER
    if observer is None:
        return
    observer.log_stage(
        event="outputs_written",
        outer_fold=None,
        model_id=None,
        run_role=None,
        stage="outputs_written",
        message="all ML output files written",
    )
    observer.finish_run(output_files=expected_output_files(), total_elapsed_sec=time.time() - run_start)


In [ ]:
def _extract_inner_cv_mae(cv_table: pd.DataFrame) -> float:
    if "MAE" not in cv_table.columns:
        raise KeyError("PyCaret pull() output must contain an 'MAE' column")
    mae_series = pd.to_numeric(cv_table["MAE"], errors="coerce").dropna()
    if mae_series.empty:
        raise ValueError("PyCaret pull() returned no valid MAE values")
    return float(mae_series.mean())


In [ ]:
def load_ml_fold_bundle(step4_root: Path, outer_fold: int) -> dict[str, object]:
    if any(
        dep is None
        for dep in (
            build_prediction_frame,
            build_pycaret_setup_kwargs,
            compute_regression_metrics,
            ensure_family_result_dir,
            load_full_feature_processed,
            save_model_artifact,
            summarize_fold_metrics,
            setup,
            create_model,
            tune_model,
            finalize_model,
            predict_model,
            pull,
        )
    ):
        initialize_runtime()

    train_df, test_df, inner_index, feature_columns = load_full_feature_processed(
        Path(step4_root),
        outer_fold=int(outer_fold),
    )
    feature_columns = list(feature_columns)
    model_train = train_df.loc[:, [*feature_columns, TARGET_COL]].copy()
    eval_df = test_df.loc[:, [*feature_columns, TARGET_COL]].copy()
    categorical_features = [column for column in feature_columns if str(column).startswith("Is_")]
    setup_kwargs = build_pycaret_setup_kwargs(
        model_train=model_train,
        target_col=TARGET_COL,
        keep_features=feature_columns,
        ignore_features=[],
        categorical_features=categorical_features,
        session_id=1100 + int(outer_fold),
    )
    return {
        "outer_fold": int(outer_fold),
        "train_df": train_df,
        "test_df": test_df,
        "inner_index": inner_index,
        "feature_columns": feature_columns,
        "model_train": model_train,
        "eval_df": eval_df,
        "categorical_features": categorical_features,
        "setup_kwargs": setup_kwargs,
    }


In [ ]:
def run_ml_baseline_model(
    fold_bundle: dict[str, object],
    model_id: str,
    engine_id: str,
    artifact_dir: Path,
    observer: MLRunObserver | None = None,
) -> dict[str, object]:
    if observer is None:
        observer = ACTIVE_ML_RUN_OBSERVER
    baseline_start = time.time()
    if observer is not None:
        observer.log_stage(
            event="baseline_start",
            outer_fold=int(fold_bundle["outer_fold"]),
            model_id=model_id,
            run_role="baseline",
            stage="baseline_start",
            message=f"starting baseline model engine={engine_id}",
        )
        observer.update_status(
            outer_fold=int(fold_bundle["outer_fold"]),
            model_id=model_id,
            run_role="baseline",
            stage="baseline_start",
            trial_index=None,
            trial_total=None,
            completed_units=observer.completed_units,
            status="running",
        )
    setup(**dict(fold_bundle["setup_kwargs"]))
    model = create_model(engine_id, fold=3, verbose=False)
    inner_cv_mae = _extract_inner_cv_mae(pull().copy())
    pred_df = predict_model(model, data=pd.DataFrame(fold_bundle["eval_df"]).copy(), verbose=False)
    prediction_frame = build_prediction_frame(
        test_df=pd.DataFrame(fold_bundle["test_df"]),
        prediction_log=pred_df["prediction_label"].to_numpy(),
        outer_fold=int(fold_bundle["outer_fold"]),
        model_family="ml",
        model_id=model_id,
        run_role="baseline",
    )
    artifact_path = artifact_dir / f"fold_{int(fold_bundle['outer_fold'])}" / f"{model_id}_baseline.joblib"
    save_model_artifact(finalize_model(model), artifact_path)
    metric_row = {
        "outer_fold": int(fold_bundle["outer_fold"]),
        "model_family": "ml",
        "model_id": model_id,
        "run_role": "baseline",
        "inner_cv_mae": inner_cv_mae,
        **compute_regression_metrics(prediction_frame),
    }
    artifact_row = {
        "outer_fold": int(fold_bundle["outer_fold"]),
        "model_family": "ml",
        "model_id": model_id,
        "run_role": "baseline",
        "artifact_path": str(artifact_path),
    }
    elapsed_sec = time.time() - baseline_start
    if observer is not None:
        observer.log_stage(
            event="baseline_end",
            outer_fold=int(fold_bundle["outer_fold"]),
            model_id=model_id,
            run_role="baseline",
            stage="baseline_end",
            message="baseline model finished",
            elapsed_sec=elapsed_sec,
        )
        observer.advance_unit(f"fold={int(fold_bundle['outer_fold'])} {model_id} baseline")
        observer.update_status(
            outer_fold=int(fold_bundle["outer_fold"]),
            model_id=model_id,
            run_role="baseline",
            stage="baseline_end",
            trial_index=None,
            trial_total=None,
            completed_units=observer.completed_units,
            status="running",
        )
    return {
        "model_id": model_id,
        "engine_id": engine_id,
        "inner_cv_mae": inner_cv_mae,
        "prediction_frame": prediction_frame,
        "metric_row": metric_row,
        "artifact_row": artifact_row,
    }


In [ ]:
def select_ml_tuned_candidates(baseline_results: list[dict[str, object]]) -> pd.DataFrame:
    if not baseline_results:
        raise RuntimeError("baseline_results must not be empty")
    return pd.DataFrame(
        [
            {
                "model_id": result["model_id"],
                "engine_id": result["engine_id"],
                "inner_cv_mae": result["inner_cv_mae"],
            }
            for result in baseline_results
        ]
    ).sort_values(
        ["inner_cv_mae", "model_id"],
        ascending=[True, True],
        kind="mergesort",
    )


In [ ]:
def run_ml_tuned_candidate(
    fold_bundle: dict[str, object],
    model_id: str,
    engine_id: str,
    artifact_dir: Path,
    observer: MLRunObserver | None = None,
) -> dict[str, object]:
    if observer is None:
        observer = ACTIVE_ML_RUN_OBSERVER
    tuned_start = time.time()
    if observer is not None:
        observer.log_stage(
            event="tuned_start",
            outer_fold=int(fold_bundle["outer_fold"]),
            model_id=model_id,
            run_role="tuned",
            stage="tuned_start",
            message=f"starting tuned model engine={engine_id}",
        )
        observer.update_status(
            outer_fold=int(fold_bundle["outer_fold"]),
            model_id=model_id,
            run_role="tuned",
            stage="tuned_start",
            trial_index=None,
            trial_total=None,
            completed_units=observer.completed_units,
            status="running",
        )
    setup(**dict(fold_bundle["setup_kwargs"]))
    model = create_model(engine_id, fold=3, verbose=False)
    _ = pull().copy()
    tuned_model = tune_model(
        model,
        n_iter=50,
        optimize="MAE",
        search_library="optuna",
        choose_better=True,
        fold=3,
        verbose=True,
        tuner_verbose=1,
    )
    inner_cv_mae = _extract_inner_cv_mae(pull().copy())
    final_model = finalize_model(tuned_model)
    pred_df = predict_model(final_model, data=pd.DataFrame(fold_bundle["eval_df"]).copy(), verbose=False)
    prediction_frame = build_prediction_frame(
        test_df=pd.DataFrame(fold_bundle["test_df"]),
        prediction_log=pred_df["prediction_label"].to_numpy(),
        outer_fold=int(fold_bundle["outer_fold"]),
        model_family="ml",
        model_id=model_id,
        run_role="tuned",
    )
    artifact_path = artifact_dir / f"fold_{int(fold_bundle['outer_fold'])}" / f"{model_id}_tuned.joblib"
    save_model_artifact(final_model, artifact_path)
    metric_row = {
        "outer_fold": int(fold_bundle["outer_fold"]),
        "model_family": "ml",
        "model_id": model_id,
        "run_role": "tuned",
        "inner_cv_mae": inner_cv_mae,
        **compute_regression_metrics(prediction_frame),
    }
    artifact_row = {
        "outer_fold": int(fold_bundle["outer_fold"]),
        "model_family": "ml",
        "model_id": model_id,
        "run_role": "tuned",
        "artifact_path": str(artifact_path),
    }
    elapsed_sec = time.time() - tuned_start
    if observer is not None:
        observer.log_stage(
            event="tuned_end",
            outer_fold=int(fold_bundle["outer_fold"]),
            model_id=model_id,
            run_role="tuned",
            stage="tuned_end",
            message="tuned model finished",
            elapsed_sec=elapsed_sec,
        )
        observer.advance_unit(f"fold={int(fold_bundle['outer_fold'])} {model_id} tuned")
        observer.update_status(
            outer_fold=int(fold_bundle["outer_fold"]),
            model_id=model_id,
            run_role="tuned",
            stage="tuned_end",
            trial_index=None,
            trial_total=None,
            completed_units=observer.completed_units,
            status="running",
        )
    return {
        "model_id": model_id,
        "engine_id": engine_id,
        "inner_cv_mae": inner_cv_mae,
        "prediction_frame": prediction_frame,
        "metric_row": metric_row,
        "artifact_row": artifact_row,
    }


In [ ]:
def write_ml_family_outputs(
    result_dir: Path,
    prediction_frames: list[pd.DataFrame],
    metric_rows: list[dict[str, object]],
    artifact_rows: list[dict[str, object]],
) -> None:
    if not prediction_frames or not metric_rows or not artifact_rows:
        raise RuntimeError("ML family compare produced no outputs")

    metrics_df = pd.DataFrame(metric_rows)
    summary_df = summarize_fold_metrics(metrics_df)
    ranking_df = summary_df.copy()
    ranking_df.insert(0, "rank", range(1, len(ranking_df) + 1))
    best_models_df = ranking_df.head(int(ML_TOP_K_TUNED)).copy()

    pd.DataFrame(artifact_rows).to_csv(result_dir / "1_ml_model_artifact_manifest.csv", index=False)
    pd.concat(prediction_frames, ignore_index=True).to_csv(
        result_dir / "1_ml_predictions_outer_test.csv",
        index=False,
    )
    metrics_df.to_csv(result_dir / "1_ml_metrics_by_model_by_outer_fold.csv", index=False)
    summary_df.to_csv(result_dir / "1_ml_metrics_by_model_mean_std.csv", index=False)
    ranking_df.to_csv(result_dir / "1_ml_model_ranking.csv", index=False)
    best_models_df.to_csv(result_dir / "1_ml_best_models.csv", index=False)


In [ ]:
def run_ml_family(step4_root: Path, step5_root: Path) -> None:
    if any(
        dep is None
        for dep in (
            build_prediction_frame,
            build_pycaret_setup_kwargs,
            compute_regression_metrics,
            ensure_family_result_dir,
            load_full_feature_processed,
            save_model_artifact,
            summarize_fold_metrics,
            setup,
            create_model,
            tune_model,
            finalize_model,
            predict_model,
            pull,
        )
    ):
        initialize_runtime()

    result_dir = ensure_family_result_dir(Path(step5_root), "ml")
    artifact_dir = result_dir / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    total_units = len(OUTER_FOLDS) * (len(ML_BASELINE_MODEL_SPECS) + int(ML_TOP_K_TUNED))
    observer = create_ml_run_observer(result_dir=result_dir, total_units=total_units)
    run_start = time.time()

    prediction_frames: list[pd.DataFrame] = []
    metric_rows: list[dict[str, object]] = []
    artifact_rows: list[dict[str, object]] = []
    current_outer_fold: int | None = None
    current_model_id: str | None = None
    current_run_role: str | None = None
    current_stage = "run_init"

    try:
        observer.start_run(step4_root=Path(step4_root), step5_root=Path(step5_root))

        for outer_fold in OUTER_FOLDS:
            current_outer_fold = int(outer_fold)
            current_model_id = None
            current_run_role = None
            current_stage = "outer_fold_start"
            observer.log_stage(
                event="outer_fold_start",
                outer_fold=int(outer_fold),
                model_id=None,
                run_role=None,
                stage="outer_fold_start",
                message="starting outer fold",
            )
            observer.update_status(
                outer_fold=int(outer_fold),
                model_id=None,
                run_role=None,
                stage="outer_fold_start",
                trial_index=None,
                trial_total=None,
                completed_units=observer.completed_units,
                status="running",
            )

            fold_bundle = load_ml_fold_bundle(step4_root=step4_root, outer_fold=int(outer_fold))
            baseline_results = []
            for model_id, engine_id in ML_BASELINE_MODEL_SPECS:
                current_model_id = model_id
                current_run_role = "baseline"
                current_stage = "baseline"
                baseline_result = run_ml_baseline_model(
                    fold_bundle,
                    model_id,
                    engine_id,
                    artifact_dir,
                    observer=observer,
                )
                baseline_results.append(baseline_result)
                prediction_frames.append(baseline_result["prediction_frame"])
                metric_rows.append(baseline_result["metric_row"])
                artifact_rows.append(baseline_result["artifact_row"])

            baseline_fold_df = select_ml_tuned_candidates(baseline_results)
            tuned_candidate_df = baseline_fold_df.head(int(ML_TOP_K_TUNED)).reset_index(drop=True)
            observer.log_stage(
                event="tuned_candidates_selected",
                outer_fold=int(outer_fold),
                model_id=",".join(tuned_candidate_df["model_id"].astype(str).tolist()),
                run_role="tuned",
                stage="tuned_candidates_selected",
                message="selected tuned candidates from baseline ranking",
            )

            for tuned_candidate in tuned_candidate_df.to_dict("records"):
                model_id = str(tuned_candidate["model_id"])
                engine_id = str(tuned_candidate["engine_id"])
                current_model_id = model_id
                current_run_role = "tuned"
                current_stage = "tuned"
                tuned_result = run_ml_tuned_candidate(
                    fold_bundle,
                    model_id,
                    engine_id,
                    artifact_dir,
                    observer=observer,
                )
                prediction_frames.append(tuned_result["prediction_frame"])
                metric_rows.append(tuned_result["metric_row"])
                artifact_rows.append(tuned_result["artifact_row"])

        current_outer_fold = None
        current_model_id = None
        current_run_role = None
        current_stage = "outputs_written"
        write_ml_family_outputs(result_dir, prediction_frames, metric_rows, artifact_rows)
        observer.log_stage(
            event="outputs_written",
            outer_fold=None,
            model_id=None,
            run_role=None,
            stage="outputs_written",
            message="all ML output files written",
        )
        observer.finish_run(output_files=expected_output_files(), total_elapsed_sec=time.time() - run_start)
    except Exception as exc:
        observer.fail_run(
            outer_fold=current_outer_fold,
            model_id=current_model_id,
            run_role=current_run_role,
            stage=current_stage,
            exc=exc,
        )
        raise


In [ ]:
# --- Step 1: Resolve release-local Step4 inputs and Step5 outputs ---
current_code_dir = initialize_runtime()
step5_root = current_code_dir.parent
release_root = step5_root.parent
step4_root = release_root / "Step4_ModelInputs" / "2_Results"
ml_result_dir = ensure_family_result_dir(step5_root, "ml")
ml_artifact_dir = ml_result_dir / "artifacts"
ml_artifact_dir.mkdir(parents=True, exist_ok=True)
ml_prediction_frames: list[pd.DataFrame] = []
ml_metric_rows: list[dict[str, object]] = []
ml_artifact_rows: list[dict[str, object]] = []
ml_total_units = len(OUTER_FOLDS) * (len(ML_BASELINE_MODEL_SPECS) + int(ML_TOP_K_TUNED))
ACTIVE_ML_RUN_OBSERVER = create_ml_run_observer(result_dir=ml_result_dir, total_units=ml_total_units)
ml_run_start = time.time()
ACTIVE_ML_RUN_OBSERVER.start_run(step4_root=step4_root, step5_root=step5_root)
print({"step": "ml_runtime_ready", "step4_root": str(step4_root), "result_dir": str(ml_result_dir)})


In [ ]:
# --- Step 2: Load fold 1 inputs ---
log_ml_mainline_outer_fold_start(1)
ml_fold_1_bundle = load_ml_fold_bundle(step4_root=step4_root, outer_fold=1)
print({"step": "ml_fold_loaded", "outer_fold": 1, "train_rows": len(ml_fold_1_bundle["train_df"]), "test_rows": len(ml_fold_1_bundle["test_df"])})


In [ ]:
# --- Step 3: Fold 1 baseline et ---
ml_fold_1_baseline_et = run_ml_baseline_model(ml_fold_1_bundle, "ml_et", "et", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_1_baseline_et["prediction_frame"])
ml_metric_rows.append(ml_fold_1_baseline_et["metric_row"])
ml_artifact_rows.append(ml_fold_1_baseline_et["artifact_row"])
print(ml_fold_1_baseline_et["metric_row"])


In [ ]:
# --- Step 4: Fold 1 baseline xgboost ---
ml_fold_1_baseline_xgboost = run_ml_baseline_model(ml_fold_1_bundle, "ml_xgboost", "xgboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_1_baseline_xgboost["prediction_frame"])
ml_metric_rows.append(ml_fold_1_baseline_xgboost["metric_row"])
ml_artifact_rows.append(ml_fold_1_baseline_xgboost["artifact_row"])
print(ml_fold_1_baseline_xgboost["metric_row"])


In [ ]:
# --- Step 5: Fold 1 baseline catboost ---
ml_fold_1_baseline_catboost = run_ml_baseline_model(ml_fold_1_bundle, "ml_catboost", "catboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_1_baseline_catboost["prediction_frame"])
ml_metric_rows.append(ml_fold_1_baseline_catboost["metric_row"])
ml_artifact_rows.append(ml_fold_1_baseline_catboost["artifact_row"])
print(ml_fold_1_baseline_catboost["metric_row"])


In [ ]:
# --- Step 6: Fold 1 baseline rf ---
ml_fold_1_baseline_rf = run_ml_baseline_model(ml_fold_1_bundle, "ml_rf", "rf", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_1_baseline_rf["prediction_frame"])
ml_metric_rows.append(ml_fold_1_baseline_rf["metric_row"])
ml_artifact_rows.append(ml_fold_1_baseline_rf["artifact_row"])
print(ml_fold_1_baseline_rf["metric_row"])


In [ ]:
# --- Step 7: Fold 1 baseline lightgbm ---
ml_fold_1_baseline_lightgbm = run_ml_baseline_model(ml_fold_1_bundle, "ml_lightgbm", "lightgbm", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_1_baseline_lightgbm["prediction_frame"])
ml_metric_rows.append(ml_fold_1_baseline_lightgbm["metric_row"])
ml_artifact_rows.append(ml_fold_1_baseline_lightgbm["artifact_row"])
print(ml_fold_1_baseline_lightgbm["metric_row"])


In [ ]:
# --- Step 8: Fold 1 select tuned candidates ---
ml_fold_1_baseline_ranking = select_ml_tuned_candidates(
    [
        ml_fold_1_baseline_et,
        ml_fold_1_baseline_xgboost,
        ml_fold_1_baseline_catboost,
        ml_fold_1_baseline_rf,
        ml_fold_1_baseline_lightgbm,
    ]
)
log_ml_mainline_tuned_candidates_selected(1, ml_fold_1_baseline_ranking)
print(ml_fold_1_baseline_ranking)


In [ ]:
# --- Step 9: Fold 1 tuned candidate 1 ---
ml_fold_1_tuned_candidate_1 = run_ml_tuned_candidate(
    ml_fold_1_bundle,
    ml_fold_1_baseline_ranking.iloc[0]["model_id"],
    ml_fold_1_baseline_ranking.iloc[0]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_1_tuned_candidate_1["prediction_frame"])
ml_metric_rows.append(ml_fold_1_tuned_candidate_1["metric_row"])
ml_artifact_rows.append(ml_fold_1_tuned_candidate_1["artifact_row"])
print(ml_fold_1_tuned_candidate_1["metric_row"])


In [ ]:
# --- Step 10: Fold 1 tuned candidate 2 ---
ml_fold_1_tuned_candidate_2 = run_ml_tuned_candidate(
    ml_fold_1_bundle,
    ml_fold_1_baseline_ranking.iloc[1]["model_id"],
    ml_fold_1_baseline_ranking.iloc[1]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_1_tuned_candidate_2["prediction_frame"])
ml_metric_rows.append(ml_fold_1_tuned_candidate_2["metric_row"])
ml_artifact_rows.append(ml_fold_1_tuned_candidate_2["artifact_row"])
print(ml_fold_1_tuned_candidate_2["metric_row"])


In [ ]:
# --- Step 11: Load fold 2 inputs ---
log_ml_mainline_outer_fold_start(2)
ml_fold_2_bundle = load_ml_fold_bundle(step4_root=step4_root, outer_fold=2)
print({"step": "ml_fold_loaded", "outer_fold": 2, "train_rows": len(ml_fold_2_bundle["train_df"]), "test_rows": len(ml_fold_2_bundle["test_df"])})


In [ ]:
# --- Step 12: Fold 2 baseline et ---
ml_fold_2_baseline_et = run_ml_baseline_model(ml_fold_2_bundle, "ml_et", "et", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_2_baseline_et["prediction_frame"])
ml_metric_rows.append(ml_fold_2_baseline_et["metric_row"])
ml_artifact_rows.append(ml_fold_2_baseline_et["artifact_row"])
print(ml_fold_2_baseline_et["metric_row"])


In [ ]:
# --- Step 13: Fold 2 baseline xgboost ---
ml_fold_2_baseline_xgboost = run_ml_baseline_model(ml_fold_2_bundle, "ml_xgboost", "xgboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_2_baseline_xgboost["prediction_frame"])
ml_metric_rows.append(ml_fold_2_baseline_xgboost["metric_row"])
ml_artifact_rows.append(ml_fold_2_baseline_xgboost["artifact_row"])
print(ml_fold_2_baseline_xgboost["metric_row"])


In [ ]:
# --- Step 14: Fold 2 baseline catboost ---
ml_fold_2_baseline_catboost = run_ml_baseline_model(ml_fold_2_bundle, "ml_catboost", "catboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_2_baseline_catboost["prediction_frame"])
ml_metric_rows.append(ml_fold_2_baseline_catboost["metric_row"])
ml_artifact_rows.append(ml_fold_2_baseline_catboost["artifact_row"])
print(ml_fold_2_baseline_catboost["metric_row"])


In [ ]:
# --- Step 15: Fold 2 baseline rf ---
ml_fold_2_baseline_rf = run_ml_baseline_model(ml_fold_2_bundle, "ml_rf", "rf", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_2_baseline_rf["prediction_frame"])
ml_metric_rows.append(ml_fold_2_baseline_rf["metric_row"])
ml_artifact_rows.append(ml_fold_2_baseline_rf["artifact_row"])
print(ml_fold_2_baseline_rf["metric_row"])

In [ ]:
# --- Step 16: Fold 2 baseline lightgbm ---
ml_fold_2_baseline_lightgbm = run_ml_baseline_model(ml_fold_2_bundle, "ml_lightgbm", "lightgbm", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_2_baseline_lightgbm["prediction_frame"])
ml_metric_rows.append(ml_fold_2_baseline_lightgbm["metric_row"])
ml_artifact_rows.append(ml_fold_2_baseline_lightgbm["artifact_row"])
print(ml_fold_2_baseline_lightgbm["metric_row"])


In [ ]:
# --- Step 17: Fold 2 select tuned candidates ---
ml_fold_2_baseline_ranking = select_ml_tuned_candidates(
    [
        ml_fold_2_baseline_et,
        ml_fold_2_baseline_xgboost,
        ml_fold_2_baseline_catboost,
        ml_fold_2_baseline_rf,
        ml_fold_2_baseline_lightgbm,
    ]
)
log_ml_mainline_tuned_candidates_selected(2, ml_fold_2_baseline_ranking)
print(ml_fold_2_baseline_ranking)


In [ ]:
# --- Step 18: Fold 2 tuned candidate 1 ---
ml_fold_2_tuned_candidate_1 = run_ml_tuned_candidate(
    ml_fold_2_bundle,
    ml_fold_2_baseline_ranking.iloc[0]["model_id"],
    ml_fold_2_baseline_ranking.iloc[0]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_2_tuned_candidate_1["prediction_frame"])
ml_metric_rows.append(ml_fold_2_tuned_candidate_1["metric_row"])
ml_artifact_rows.append(ml_fold_2_tuned_candidate_1["artifact_row"])
print(ml_fold_2_tuned_candidate_1["metric_row"])


In [ ]:
# --- Step 19: Fold 2 tuned candidate 2 ---
ml_fold_2_tuned_candidate_2 = run_ml_tuned_candidate(
    ml_fold_2_bundle,
    ml_fold_2_baseline_ranking.iloc[1]["model_id"],
    ml_fold_2_baseline_ranking.iloc[1]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_2_tuned_candidate_2["prediction_frame"])
ml_metric_rows.append(ml_fold_2_tuned_candidate_2["metric_row"])
ml_artifact_rows.append(ml_fold_2_tuned_candidate_2["artifact_row"])
print(ml_fold_2_tuned_candidate_2["metric_row"])


In [ ]:
# --- Step 20: Load fold 3 inputs ---
log_ml_mainline_outer_fold_start(3)
ml_fold_3_bundle = load_ml_fold_bundle(step4_root=step4_root, outer_fold=3)
print({"step": "ml_fold_loaded", "outer_fold": 3, "train_rows": len(ml_fold_3_bundle["train_df"]), "test_rows": len(ml_fold_3_bundle["test_df"])})


In [ ]:
# --- Step 21: Fold 3 baseline et ---
ml_fold_3_baseline_et = run_ml_baseline_model(ml_fold_3_bundle, "ml_et", "et", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_3_baseline_et["prediction_frame"])
ml_metric_rows.append(ml_fold_3_baseline_et["metric_row"])
ml_artifact_rows.append(ml_fold_3_baseline_et["artifact_row"])
print(ml_fold_3_baseline_et["metric_row"])


In [ ]:
# --- Step 22: Fold 3 baseline xgboost ---
ml_fold_3_baseline_xgboost = run_ml_baseline_model(ml_fold_3_bundle, "ml_xgboost", "xgboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_3_baseline_xgboost["prediction_frame"])
ml_metric_rows.append(ml_fold_3_baseline_xgboost["metric_row"])
ml_artifact_rows.append(ml_fold_3_baseline_xgboost["artifact_row"])
print(ml_fold_3_baseline_xgboost["metric_row"])


In [ ]:
# --- Step 23: Fold 3 baseline catboost ---
ml_fold_3_baseline_catboost = run_ml_baseline_model(ml_fold_3_bundle, "ml_catboost", "catboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_3_baseline_catboost["prediction_frame"])
ml_metric_rows.append(ml_fold_3_baseline_catboost["metric_row"])
ml_artifact_rows.append(ml_fold_3_baseline_catboost["artifact_row"])
print(ml_fold_3_baseline_catboost["metric_row"])


In [ ]:
# --- Step 24: Fold 3 baseline rf ---
ml_fold_3_baseline_rf = run_ml_baseline_model(ml_fold_3_bundle, "ml_rf", "rf", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_3_baseline_rf["prediction_frame"])
ml_metric_rows.append(ml_fold_3_baseline_rf["metric_row"])
ml_artifact_rows.append(ml_fold_3_baseline_rf["artifact_row"])
print(ml_fold_3_baseline_rf["metric_row"])


In [ ]:
# --- Step 25: Fold 3 baseline lightgbm ---
ml_fold_3_baseline_lightgbm = run_ml_baseline_model(ml_fold_3_bundle, "ml_lightgbm", "lightgbm", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_3_baseline_lightgbm["prediction_frame"])
ml_metric_rows.append(ml_fold_3_baseline_lightgbm["metric_row"])
ml_artifact_rows.append(ml_fold_3_baseline_lightgbm["artifact_row"])
print(ml_fold_3_baseline_lightgbm["metric_row"])


In [ ]:
# --- Step 26: Fold 3 select tuned candidates ---
ml_fold_3_baseline_ranking = select_ml_tuned_candidates(
    [
        ml_fold_3_baseline_et,
        ml_fold_3_baseline_xgboost,
        ml_fold_3_baseline_catboost,
        ml_fold_3_baseline_rf,
        ml_fold_3_baseline_lightgbm,
    ]
)
log_ml_mainline_tuned_candidates_selected(3, ml_fold_3_baseline_ranking)
print(ml_fold_3_baseline_ranking)


In [ ]:
# --- Step 27: Fold 3 tuned candidate 1 ---
ml_fold_3_tuned_candidate_1 = run_ml_tuned_candidate(
    ml_fold_3_bundle,
    ml_fold_3_baseline_ranking.iloc[0]["model_id"],
    ml_fold_3_baseline_ranking.iloc[0]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_3_tuned_candidate_1["prediction_frame"])
ml_metric_rows.append(ml_fold_3_tuned_candidate_1["metric_row"])
ml_artifact_rows.append(ml_fold_3_tuned_candidate_1["artifact_row"])
print(ml_fold_3_tuned_candidate_1["metric_row"])


In [ ]:
# --- Step 28: Fold 3 tuned candidate 2 ---
ml_fold_3_tuned_candidate_2 = run_ml_tuned_candidate(
    ml_fold_3_bundle,
    ml_fold_3_baseline_ranking.iloc[1]["model_id"],
    ml_fold_3_baseline_ranking.iloc[1]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_3_tuned_candidate_2["prediction_frame"])
ml_metric_rows.append(ml_fold_3_tuned_candidate_2["metric_row"])
ml_artifact_rows.append(ml_fold_3_tuned_candidate_2["artifact_row"])
print(ml_fold_3_tuned_candidate_2["metric_row"])


In [ ]:
# --- Step 29: Load fold 4 inputs ---
log_ml_mainline_outer_fold_start(4)
ml_fold_4_bundle = load_ml_fold_bundle(step4_root=step4_root, outer_fold=4)
print({"step": "ml_fold_loaded", "outer_fold": 4, "train_rows": len(ml_fold_4_bundle["train_df"]), "test_rows": len(ml_fold_4_bundle["test_df"])})


In [ ]:
# --- Step 30: Fold 4 baseline et ---
ml_fold_4_baseline_et = run_ml_baseline_model(ml_fold_4_bundle, "ml_et", "et", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_4_baseline_et["prediction_frame"])
ml_metric_rows.append(ml_fold_4_baseline_et["metric_row"])
ml_artifact_rows.append(ml_fold_4_baseline_et["artifact_row"])
print(ml_fold_4_baseline_et["metric_row"])


In [ ]:
# --- Step 31: Fold 4 baseline xgboost ---
ml_fold_4_baseline_xgboost = run_ml_baseline_model(ml_fold_4_bundle, "ml_xgboost", "xgboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_4_baseline_xgboost["prediction_frame"])
ml_metric_rows.append(ml_fold_4_baseline_xgboost["metric_row"])
ml_artifact_rows.append(ml_fold_4_baseline_xgboost["artifact_row"])
print(ml_fold_4_baseline_xgboost["metric_row"])


In [ ]:
# --- Step 32: Fold 4 baseline catboost ---
ml_fold_4_baseline_catboost = run_ml_baseline_model(ml_fold_4_bundle, "ml_catboost", "catboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_4_baseline_catboost["prediction_frame"])
ml_metric_rows.append(ml_fold_4_baseline_catboost["metric_row"])
ml_artifact_rows.append(ml_fold_4_baseline_catboost["artifact_row"])
print(ml_fold_4_baseline_catboost["metric_row"])


In [ ]:
# --- Step 33: Fold 4 baseline rf ---
ml_fold_4_baseline_rf = run_ml_baseline_model(ml_fold_4_bundle, "ml_rf", "rf", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_4_baseline_rf["prediction_frame"])
ml_metric_rows.append(ml_fold_4_baseline_rf["metric_row"])
ml_artifact_rows.append(ml_fold_4_baseline_rf["artifact_row"])
print(ml_fold_4_baseline_rf["metric_row"])


In [ ]:
# --- Step 34: Fold 4 baseline lightgbm ---
ml_fold_4_baseline_lightgbm = run_ml_baseline_model(ml_fold_4_bundle, "ml_lightgbm", "lightgbm", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_4_baseline_lightgbm["prediction_frame"])
ml_metric_rows.append(ml_fold_4_baseline_lightgbm["metric_row"])
ml_artifact_rows.append(ml_fold_4_baseline_lightgbm["artifact_row"])
print(ml_fold_4_baseline_lightgbm["metric_row"])


In [ ]:
# --- Step 35: Fold 4 select tuned candidates ---
ml_fold_4_baseline_ranking = select_ml_tuned_candidates(
    [
        ml_fold_4_baseline_et,
        ml_fold_4_baseline_xgboost,
        ml_fold_4_baseline_catboost,
        ml_fold_4_baseline_rf,
        ml_fold_4_baseline_lightgbm,
    ]
)
log_ml_mainline_tuned_candidates_selected(4, ml_fold_4_baseline_ranking)
print(ml_fold_4_baseline_ranking)


In [ ]:
# --- Step 36: Fold 4 tuned candidate 1 ---
ml_fold_4_tuned_candidate_1 = run_ml_tuned_candidate(
    ml_fold_4_bundle,
    ml_fold_4_baseline_ranking.iloc[0]["model_id"],
    ml_fold_4_baseline_ranking.iloc[0]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_4_tuned_candidate_1["prediction_frame"])
ml_metric_rows.append(ml_fold_4_tuned_candidate_1["metric_row"])
ml_artifact_rows.append(ml_fold_4_tuned_candidate_1["artifact_row"])
print(ml_fold_4_tuned_candidate_1["metric_row"])


In [ ]:
# --- Step 37: Fold 4 tuned candidate 2 ---
ml_fold_4_tuned_candidate_2 = run_ml_tuned_candidate(
    ml_fold_4_bundle,
    ml_fold_4_baseline_ranking.iloc[1]["model_id"],
    ml_fold_4_baseline_ranking.iloc[1]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_4_tuned_candidate_2["prediction_frame"])
ml_metric_rows.append(ml_fold_4_tuned_candidate_2["metric_row"])
ml_artifact_rows.append(ml_fold_4_tuned_candidate_2["artifact_row"])
print(ml_fold_4_tuned_candidate_2["metric_row"])


In [ ]:
# --- Step 38: Load fold 5 inputs ---
log_ml_mainline_outer_fold_start(5)
ml_fold_5_bundle = load_ml_fold_bundle(step4_root=step4_root, outer_fold=5)
print({"step": "ml_fold_loaded", "outer_fold": 5, "train_rows": len(ml_fold_5_bundle["train_df"]), "test_rows": len(ml_fold_5_bundle["test_df"])})


In [ ]:
# --- Step 39: Fold 5 baseline et ---
ml_fold_5_baseline_et = run_ml_baseline_model(ml_fold_5_bundle, "ml_et", "et", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_5_baseline_et["prediction_frame"])
ml_metric_rows.append(ml_fold_5_baseline_et["metric_row"])
ml_artifact_rows.append(ml_fold_5_baseline_et["artifact_row"])
print(ml_fold_5_baseline_et["metric_row"])


In [ ]:
# --- Step 40: Fold 5 baseline xgboost ---
ml_fold_5_baseline_xgboost = run_ml_baseline_model(ml_fold_5_bundle, "ml_xgboost", "xgboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_5_baseline_xgboost["prediction_frame"])
ml_metric_rows.append(ml_fold_5_baseline_xgboost["metric_row"])
ml_artifact_rows.append(ml_fold_5_baseline_xgboost["artifact_row"])
print(ml_fold_5_baseline_xgboost["metric_row"])


In [ ]:
# --- Step 41: Fold 5 baseline catboost ---
ml_fold_5_baseline_catboost = run_ml_baseline_model(ml_fold_5_bundle, "ml_catboost", "catboost", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_5_baseline_catboost["prediction_frame"])
ml_metric_rows.append(ml_fold_5_baseline_catboost["metric_row"])
ml_artifact_rows.append(ml_fold_5_baseline_catboost["artifact_row"])
print(ml_fold_5_baseline_catboost["metric_row"])


In [ ]:
# --- Step 42: Fold 5 baseline rf ---
ml_fold_5_baseline_rf = run_ml_baseline_model(ml_fold_5_bundle, "ml_rf", "rf", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_5_baseline_rf["prediction_frame"])
ml_metric_rows.append(ml_fold_5_baseline_rf["metric_row"])
ml_artifact_rows.append(ml_fold_5_baseline_rf["artifact_row"])
print(ml_fold_5_baseline_rf["metric_row"])


In [ ]:
# --- Step 43: Fold 5 baseline lightgbm ---
ml_fold_5_baseline_lightgbm = run_ml_baseline_model(ml_fold_5_bundle, "ml_lightgbm", "lightgbm", ml_artifact_dir)
ml_prediction_frames.append(ml_fold_5_baseline_lightgbm["prediction_frame"])
ml_metric_rows.append(ml_fold_5_baseline_lightgbm["metric_row"])
ml_artifact_rows.append(ml_fold_5_baseline_lightgbm["artifact_row"])
print(ml_fold_5_baseline_lightgbm["metric_row"])


In [ ]:
# --- Step 44: Fold 5 select tuned candidates ---
ml_fold_5_baseline_ranking = select_ml_tuned_candidates(
    [
        ml_fold_5_baseline_et,
        ml_fold_5_baseline_xgboost,
        ml_fold_5_baseline_catboost,
        ml_fold_5_baseline_rf,
        ml_fold_5_baseline_lightgbm,
    ]
)
log_ml_mainline_tuned_candidates_selected(5, ml_fold_5_baseline_ranking)
print(ml_fold_5_baseline_ranking)


In [ ]:
# --- Step 45: Fold 5 tuned candidate 1 ---
ml_fold_5_tuned_candidate_1 = run_ml_tuned_candidate(
    ml_fold_5_bundle,
    ml_fold_5_baseline_ranking.iloc[0]["model_id"],
    ml_fold_5_baseline_ranking.iloc[0]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_5_tuned_candidate_1["prediction_frame"])
ml_metric_rows.append(ml_fold_5_tuned_candidate_1["metric_row"])
ml_artifact_rows.append(ml_fold_5_tuned_candidate_1["artifact_row"])
print(ml_fold_5_tuned_candidate_1["metric_row"])


In [ ]:
# --- Step 46: Fold 5 tuned candidate 2 ---
ml_fold_5_tuned_candidate_2 = run_ml_tuned_candidate(
    ml_fold_5_bundle,
    ml_fold_5_baseline_ranking.iloc[1]["model_id"],
    ml_fold_5_baseline_ranking.iloc[1]["engine_id"],
    ml_artifact_dir,
)
ml_prediction_frames.append(ml_fold_5_tuned_candidate_2["prediction_frame"])
ml_metric_rows.append(ml_fold_5_tuned_candidate_2["metric_row"])
ml_artifact_rows.append(ml_fold_5_tuned_candidate_2["artifact_row"])
print(ml_fold_5_tuned_candidate_2["metric_row"])


In [ ]:
# --- Step 47: Write ML family outputs ---
finalize_ml_mainline_run(ml_result_dir, ml_prediction_frames, ml_metric_rows, ml_artifact_rows, ml_run_start)
print({"step": "ml_outputs_written", "files": expected_output_files(), "metric_rows": len(ml_metric_rows)})
